## Complements

In [1]:
%run ../utils/pairwise.py
%run ../utils/sampling.py
%run ../utils/complements.py
import pandas as pd

# Load files
all_orders_df = pd.read_csv('../data/cleaned/order-products-full.csv')
products_df = pd.read_csv('../data/cleaned/product-info-full.csv')

file_path_base = "../data/validation/complements/"

# Get list of sampled products and split orders into train and test sets
sampled_products, train_df, test_df = sample_products_and_split_orders(
    products_df,
    all_orders_df,
    target_sample_size=5000,
    min_orders=50,
    test_size=0.2,
    random_state=42
)

# Simplify train orders df
order_product_df = train_df[['order_id', 'product_id']]

# Compute probabilities for the train set
product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    sampled_products,
    output_csv=f"{file_path_base}train-pairwise.csv",
    batch_size=1000
)

# Simplify test orders df
order_product_test_df = test_df[['order_id', 'product_id']]

# Compute probabilities for the test set
product_pair_test_df = compute_pairwise_probabilities_sample(order_product_test_df,
    sampled_products,
    output_csv=f"{file_path_base}test-pairwise.csv",
    batch_size=1000
)

Sampled 5000 products from 26686 eligible products.
Train orders: 5049729 rows, Test orders: 1264861 rows


100%|██████████| 5/5 [00:31<00:00,  6.35s/it]


Completed computation. Saved to ../data/validation/complements/train-pairwise.csv


100%|██████████| 5/5 [00:15<00:00,  3.04s/it]

Completed computation. Saved to ../data/validation/complements/test-pairwise.csv


In [10]:
import pandas as pd
import numpy as np


# Load dataframes
pairwise_train_df = pd.read_csv(f"{file_path_base}train-pairwise.csv")
pairwise_test_df = pd.read_csv(f"{file_path_base}test-pairwise.csv")
orders_test_df = order_product_test_df.copy()


# Parameters
top_n_complements = 10
alpha_weights = {'lift':0.4,'centrality':0.3,'cohesion':0.2,'cross_cluster':0.1}
num_samples = 3  # number of random subsets for robustness
sample_fraction = 0.2  # fraction of products per sample

# List of unique products
all_products = pd.concat([pairwise_train_df['product_i'], pairwise_train_df['product_j']]).unique()

for i in range(num_samples):
    print(f"\n--- Sample {i+1} ---")
    
    # Sample a subset of products
    sampled_products = np.random.choice(all_products, size=int(len(all_products)*sample_fraction), replace=False)
    
    # Step 1: Compute lift
    lift_df = compute_lift(sampled_products, pairwise_train_df)
    lift_df.to_csv(f"{file_path_base}sample-{i}-lift.csv", index=False)
    
    # Step 2: Compute complements + CII
    complements_df = compute_complements_and_cii(lift_df, alpha=alpha_weights)
    complements_df.to_csv(f"{file_path_base}sample-{i}-complements.csv", index=False)




--- Sample 1 ---

--- Sample 2 ---

--- Sample 3 ---


In [2]:
%run ../utils/complements.py
import pandas as pd
import numpy as np


# Load dataframes
pairwise_train_df = pd.read_csv(f"{file_path_base}train-pairwise.csv")
pairwise_test_df = pd.read_csv(f"{file_path_base}test-pairwise.csv")
orders_test_df = order_product_test_df.copy()

results = run_samples_example(
    pairwise_train_df,
    pairwise_test_df,
     orders_test_df
)

results.to_csv(f"{file_path_base}results", index=False)


--- Sample 1/3 (size=100) ---
compute_lift: 1.6s, edges: 80458
compute_complements_and_cii: 215.6s, complements: 24139
Validation (sample 1) summary: {'precision@N': 0.17574246694125298, 'recall@N': 0.003611385231527196, 'hit_rate': 0.39128549750704533, 'coverage_fraction': 1.0}

--- Sample 2/3 (size=100) ---
compute_lift: 1.7s, edges: 86072
compute_complements_and_cii: 254.9s, complements: 24604
Validation (sample 2) summary: {'precision@N': 0.17228141203210454, 'recall@N': 0.003607201985401781, 'hit_rate': 0.37715746857021093, 'coverage_fraction': 1.0}

--- Sample 3/3 (size=100) ---
compute_lift: 1.9s, edges: 109830
compute_complements_and_cii: 276.4s, complements: 28792
Validation (sample 3) summary: {'precision@N': 0.20281344932339135, 'recall@N': 0.004899583404821428, 'hit_rate': 0.44304059652029826, 'coverage_fraction': 1.0}


AttributeError: 'list' object has no attribute 'to_csv'

In [ ]:
%run ../utils/complements.py
import pandas as pd
import numpy as np


# Load dataframes
pairwise_train_df = pd.read_csv(f"{file_path_base}train-pairwise.csv")
pairwise_test_df = pd.read_csv(f"{file_path_base}test-pairwise.csv")
orders_test_df = order_product_test_df.copy()


for i in range(num_samples):
    print(f"\n--- Sample {i+1} ---")
    
    # Load dataframes
    lift_df = pd.read_csv(f"{file_path_base}sample-{i}-lift.csv")
    complements_df = pd.read_csv(f"{file_path_base}sample-{i}-complements.csv")

    complements_df = complements_df.merge(
        lift_df[['product_i','product_j','P_ij']],
        left_on=['product_id','complement_id'],
        right_on=['product_i','product_j'],
        how='left'
    ).drop(columns=['product_i','product_j'])

    # Validate results
    """  validation_results = validate_complements_pipeline(
        complements_df=complements_df,
        lift_df=lift_df,
        orders_test_df=orders_test_df,
        top_n=top_n_complements,
        alpha=alpha_weights,
        track_runtime=True
    ) """

    run_samples_example(
        pairwise_train_df,
        pairwise_test_df,
        orders_test_df
    )

    
    all_results.append(validation_results)


all_results.to_csv(f"{file_path_base}results", index=False)

# Aggregate results
mean_precision = np.mean([res['temporal']['precision@N'] for res in all_results])
mean_recall = np.mean([res['temporal']['recall@N'] for res in all_results])
print(f"\nAverage Precision@{top_n_complements} across samples: {mean_precision:.3f}")
print(f"Average Recall@{top_n_complements} across samples: {mean_recall:.3f}")


--- Sample 1 ---
{'temporal': {'precision@N': 0.17510758271562593, 'recall@N': 0.01251300652760252, 'hit_rate': 0.662264905962385}}
{'temporal': {'precision@N': 0.17510758271562593, 'recall@N': 0.01251300652760252, 'hit_rate': 0.662264905962385}, 'correlations': {'CII_vs_total_lift': 0.5936683151976049, 'CII_vs_neighbors': 0.7439022067160086}}


KeyboardInterrupt: 